# **Clasificación de textos científicos en Esperanto**

**Minería de Textos · CUNEF Universidad**

**Integrantes:** Gonzalo Casado Cabezas, Hugo Cortazar Alejo y Nicolás Mangas Álvarez.

Nuestro objetivo es clasificar párrafos en **biología (biologio), química (kemio) y física (fiziko)**. Elegimos estas disciplinas porque comparten vocabulario científico, lo que dificulta distinguirlas.

Recogemos textos de Wikipedia, construimos un NLP propio y entrenamos un clasificador que después aplicamos a páginas web reales.

En Google Colab usamos CPU y la opción **Ejecutar todas**. Por defecto cargamos los CSV de partida de nuestro repositorio público (definido en la configuración); también podemos regenerar los datos desde Wikipedia. Usamos scikit-learn 1.8.0 para reproducir la evaluación.

Ampliamos las fuentes de 36 a 62 artículos para estudiar el efecto de la diversidad del corpus sobre la generalización.


In [ ]:
# Instalamos la versión de referencia solo si no está disponible.
from importlib.metadata import version as version_instalada
import subprocess
import sys

if version_instalada("scikit-learn") != "1.8.0":
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn==1.8.0"],
                       check=True, timeout=120)
    except (subprocess.SubprocessError, OSError) as error:
        print(f"No se pudo instalar la versión de referencia: {error}")


In [ ]:
from importlib.metadata import version
from urllib.parse import quote
from IPython.display import display
import sys
import pandas as pd
import sklearn
import time

# Comprobamos la versión que se ha cargado realmente.
if sklearn.__version__ != "1.8.0":
    print(f"Aviso: usamos scikit-learn {sklearn.__version__}; las particiones y métricas pueden cambiar.")

# Fijamos la semilla y definimos las categorías y la fuente del corpus.
SEED = 42
LABELS = ["biologio", "kemio", "fiziko"]
REPO = "gcasadocabezas/Proyecto-Miner-a-de-Texto-"
REF = "main"
raw_url = f"https://raw.githubusercontent.com/{REPO}/{REF}"

# Mostramos las versiones para poder reproducir la ejecución.
print("Versiones:", ", ".join(
    f"{name} {version(name)}" for name in ("pandas", "beautifulsoup4", "matplotlib")
), f"· scikit-learn {sklearn.__version__} · Python {sys.version.split()[0]}")


## **1. Construcción del corpus**

Seleccionamos artículos de Wikipedia en Esperanto y asignamos a cada párrafo la categoría de su artículo. Quitamos secciones auxiliares, enlaces, referencias, duplicados y textos de menos de 60 caracteres. Esta longitud es un filtro práctico, no una garantía de calidad lingüística.

El dataset tiene dos columnas (`text;class`), codificación UTF-8 y separador `;`. En otro CSV guardamos la procedencia: artículo, revisión de Wikipedia y huella SHA-256 del texto. La huella permite recuperar cada párrafo sin depender de su posición en el archivo.


En la revisión final sustituimos dos páginas de biología: `Ĉelo`, que es una desambiguación, y `Evoluismo`, centrada en el uso del término, por `Ĉelo (biologio)` y `Evolucio (teorio)`. El resto de artículos se conserva cuando usamos el corpus publicado.

In [ ]:
# Artículos originales y ampliación, con los títulos canónicos de Wikipedia.
ARTICULOS_ORIGINALES = {'biologio': ['Biologio',
              'Genetiko',
              'Fotosintezo',
              'Zoologio',
              'Plantoscienco',
              'Mikrobiologio',
              'Ekologio',
              'DNA',
              'Anatomio',
              'Biodiverseco',
              'Metabolo',
              'Ĉelo (biologio)',
              'Evolucio (teorio)'],
 'kemio': ['Kemio',
           'Atomo',
           'Perioda tabelo',
           'Ĥemia ligo',
           'Organika kemio',
           'Neorganika ĥemio',
           'Ĥemia reakcio',
           'Acido',
           'Molekulo',
           'Katalizo'],
 'fiziko': ['Fiziko',
            'Klasika mekaniko',
            'Termodinamiko',
            'Elektromagnetismo',
            'Kvantuma mekaniko',
            'Teorio de la relativeco',
            'Optiko',
            'Kineta energio',
            'Partikla fiziko',
            'Astrofiziko',
            'Gravito',
            'Akustiko',
            'Nuklea fiziko']}

ARTICULOS_NUEVOS = {'biologio': ['Imuna sistemo',
              'Nerva sistemo',
              'Bakterioj',
              'Fungoj',
              'RNA',
              'Mejozo',
              'Endokrina sistemo'],
 'kemio': ['Redoksa reakcio',
           'Elektrolizo',
           'Bazo (ĥemio)',
           'Salo (ĥemio)',
           'Alkoholo',
           'Ĥemia ekvilibro',
           'Analiza ĥemio',
           'Karbono',
           'Oksigeno',
           'Nitrogeno'],
 'fiziko': ['Elektra kurento',
            'Elektromagneta radiado',
            'Sono',
            'Lumo',
            'Fluidodinamiko',
            'Premo',
            'Viskozeco',
            'Movokvanto',
            'Forto']}

CATEGORIAS = {clase: ARTICULOS_ORIGINALES[clase] + ARTICULOS_NUEVOS[clase] for clase in LABELS}

# Excluimos las secciones que normalmente no contienen texto temático.
SECCIONES_RUIDOSAS = {
    "referencoj",
    "referencoj kaj notoj",
    "eksteraj ligiloj",
    "vidu ankaŭ",
    "vidu ankau",
    "bibliografio",
    "literaturo",
    "notoj",
    "fontoj", "en esperanto", "en esperanto aperis", "faklibroj en esperanto",
    "bildaro", "galerio", "organizoj", "personoj", "informiloj",
    "libroj", "gazetoj", "filmoj", "ligiloj", "ekstera ligilo", "aliaj projektoj",
    "proverbo", "proverboj", "en arto", "en lingvo", "figura uzo",
    "pensoj pri lumo en la homa socio"
}

# Reservamos las páginas de demostración web, también durante la ampliación.
PAGINAS_WEB = {"Ribosomo": "biologio", "Hidrogeno": "kemio", "Magneta kampo": "fiziko"}


In [ ]:
"""Construcción y lectura del corpus de párrafos de Wikipedia en Esperanto."""

import hashlib
import json
import logging
import re
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from email.utils import parsedate_to_datetime
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from io import BytesIO

import pandas as pd


DATASET_CSV = "esperanto_scienco_dataset.csv"
SOURCES_CSV = "esperanto_scienco_fuentes.csv"
DATASET_COLUMNS = ["text", "class"]
SOURCE_COLUMNS = ["text_sha256", "article_id", "article_title", "article_revision"]
WIKIPEDIA_API = "https://eo.wikipedia.org/w/api.php"
USER_AGENT = f"MineriaTextosEsperanto/1.0 (https://github.com/{REPO}/)"
LOGGER = logging.getLogger(__name__)


@dataclass(frozen=True)
class Article:
    """Texto y procedencia de una página de Wikipedia."""

    page_id: int
    title: str
    revision: int
    text: str


def retry_delay(value, attempt):
    """Lee Retry-After en segundos o en formato de fecha HTTP."""
    fallback = 2 ** (attempt + 1)
    if value is None:
        return fallback
    try:
        return max(fallback, float(value))
    except ValueError:
        try:
            date = parsedate_to_datetime(value)
        except (TypeError, ValueError, OverflowError):
            return fallback
        if date.tzinfo is None:
            date = date.replace(tzinfo=timezone.utc)
        return max(fallback, (date - datetime.now(timezone.utc)).total_seconds())


def fetch_article(title, attempts=3):
    """Devuelve una página y su revisión, o None al agotar los reintentos."""
    # Construimos la consulta y pedimos texto plano y datos de la revisión.
    query = urlencode({
        "action": "query", "prop": "extracts|info|pageprops", "explaintext": 1,
        "redirects": 1, "titles": title, "format": "json", "formatversion": 2,
    })
    request = Request(
        f"{WIKIPEDIA_API}?{query}",
        headers={"User-Agent": USER_AGENT, "Accept": "application/json"},
    )

    # Reintentamos los fallos temporales sin bloquear el resto de artículos.
    for attempt in range(attempts):
        try:
            with urlopen(request, timeout=10) as response:
                payload = json.load(response)
            pages = payload.get("query", {}).get("pages", [])
            if pages and pages[0].get("extract", "").strip():
                page = pages[0]
                if "disambiguation" in page.get("pageprops", {}):
                    LOGGER.warning("Se omite la página de desambiguación %s", title)
                    return None
                return Article(page["pageid"], page["title"], page["lastrevid"], page["extract"])
            LOGGER.warning("Wikipedia no devolvió texto para %s", title)
            return None
        except HTTPError as error:
            if error.code not in (429, 500, 502, 503, 504):
                LOGGER.warning("HTTP %s en %s", error.code, title)
                return None
            wait = retry_delay(error.headers.get("Retry-After"), attempt)
        except (URLError, TimeoutError, OSError, ValueError, KeyError, TypeError, AttributeError) as error:
            LOGGER.warning("Error al leer %s: %s", title, error)
            wait = retry_delay(None, attempt)

        if attempt + 1 == attempts or wait > 20:
            break
        time.sleep(wait)

    LOGGER.warning("Se omite %s tras un fallo persistente", title)
    return None


# Identificamos algunas referencias residuales del CSV original.
# Excluimos estas citas por su formato, no por su temática.
CITAS_RESIDUALES = (
    "Berlow EL (1999) Strong effects of weak interactions",
    "Alain Milon, Pour une critique de la raison écologique",
    "Eugene P. Odum: Fundamentals of ecology",
    "Ludwig Trepl: Allgemeine Ökologie",
    "Ecology and Society - A journal of integrative science",
)


# Residuos identificados al revisar las secciones auxiliares.
RESIDUOS_AUXILIARES = (
    'NEERGAAARD, Paul, Scienco kaj pseŭdoscienco',
    'ROSTAND, Jean, La nuna stato',
    'ESPERANTLINGVA KRESTOMATIO',
    'D-ro Zoltán Varga: Gvidlibro',
    'Lamarko (Lamarck), Filozofio zoologia',
    '(Bonvolu meti ĉi tien ligilojn',
    'Kontraŭargumento: Vikipedio servu',
    'En tiu ĉi artikolo estas uzita traduko',
    'Medicina Esperantista Grupo',
    '- La gazeto Unesko Kuriero',
    'Peter Bates kaj Prasert',
    'Intervjuo kun Edward Norton',
    'Intervjuo kun Remco van Merm',
    'Dena R. Spatz kaj Nick D. Holmes',
    'Loïc Chauveau, Urboj',
    'Katerina Markelova, Titikako',
    'Dara Greenwood, Kion niaj',
    'Intervjuo kun Olivette Otele',
    'Ĥemiaj elementoj, 5-lingva tabelo',
    'La terminologia vortaro Ĥemiaj Afiksoj',
    'Scienca kaj Teknika Esperanto-Biblioteko',
    'Terminaro pri Doplera efiko',
    'Carlos Spínola en Veturado',
    # Párrafos culturales, lingüísticos y proverbios de Lumo, Sono, Acido y Movokvanto.
    'Ofte la koncepto de lumo havas asociojn',
    'Lumo de la Mondo (en la greka',
    'La movado de la Lumoj (Lumières)',
    'Tiurilate El siglo de las luces',
    'Ekzistas 6 proverboj pri ombro',
    'En fonetiko kaj fonologio, sonoranto',
    'Sonorantoj estas tiuj artikulacioj',
    'Registro, aŭ voĉregistro',
    'Sonarto (aŭ arto de sono)',
    'Muziko (greke',
    'La sono, kombine kun la silento',
    'Ekzistas pluraj proverboj pri sono',
    'Ekzistas proverboj pri acido',
    'Oni diras ke procezo "akiras movokvanto',
)

def limpiar_texto(paragraph):
    """Elimina marcas residuales y descarta referencias bibliográficas identificadas."""
    if re.search(r"\b(?:ISBN|ISSN)\b", paragraph, flags=re.IGNORECASE):
        return ""
    # Descartamos también las listas de enlaces del tipo «Véase también».
    if re.match(r"^\s*vidu\s+anka[ŭu]\s*[:：]", paragraph, flags=re.IGNORECASE):
        return ""
    if paragraph.startswith(CITAS_RESIDUALES):
        return ""
    # Conservamos el texto visible de los enlaces wiki y quitamos las marcas.
    paragraph = re.sub(r"\[{2,}[^\]\|]+\|([^\]]+)\]{2,}", r"\1", paragraph)
    paragraph = re.sub(r"\[\d+\]|https?://\S+|www\.\S+|<[^>]+>", " ", paragraph)
    # Quitamos también las marcas de enlaces wiki sin texto alternativo.
    paragraph = re.sub(r"\[\[([^\]]+)\]\]", r"\1", paragraph)
    return re.sub(r"\s+", " ", paragraph.replace(";", ",")).strip()


def es_seccion_ruidosa(titulo):
    # Incluimos variantes de encabezados como Ligiloj (anglalingve).
    return any(titulo == seccion or titulo.startswith(seccion + " ")
               or titulo.startswith(seccion + ":") for seccion in SECCIONES_RUIDOSAS)

def clean_paragraphs(text, minimum_length=60):
    """Extrae párrafos útiles de un extracto de texto de MediaWiki."""
    # Recorremos los bloques de texto y omitimos las secciones auxiliares.
    omit_level = None
    for line in text.splitlines():
        paragraph = line.strip()
        heading = re.fullmatch(r"(={2,})\s*(.*?)\s*\1", paragraph)
        plain_heading = paragraph.lower().strip(" .:")
        if heading:
            title = heading[2].lower().strip()
            nivel = len(heading[1])
            if omit_level is not None and nivel > omit_level:
                continue
            omit_level = nivel if es_seccion_ruidosa(title) else None
            continue
        if plain_heading in SECCIONES_RUIDOSAS or ("(" in plain_heading and len(plain_heading) < 70 and es_seccion_ruidosa(plain_heading)):
            omit_level = 2
            continue
        if omit_level is not None or not paragraph:
            continue

        paragraph = limpiar_texto(paragraph)
        if len(paragraph) >= minimum_length and "wayback machine" not in paragraph.casefold():
            yield paragraph


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def collect_dataset(categories=CATEGORIAS, fetch=fetch_article, pause=1, minimum_per_class=30):
    """Recoge el corpus y su procedencia, manteniendo los fallos por artículo aislados."""
    rows = []
    for label, titles in categories.items():
        for title in titles:
            article = fetch(title)
            if article is not None:
                rows.extend(
                    (paragraph, label, article.page_id, article.title, article.revision)
                    for paragraph in clean_paragraphs(article.text)
                )
            time.sleep(pause)

    corpus = pd.DataFrame(rows, columns=DATASET_COLUMNS + SOURCE_COLUMNS[1:])
    # Quitamos duplicados y textos con etiquetas contradictorias.
    label_counts = corpus.groupby("text")["class"].transform("nunique")
    corpus = corpus[label_counts == 1].drop_duplicates("text").reset_index(drop=True)
    counts = corpus["class"].value_counts().reindex(categories, fill_value=0)
    if (counts < minimum_per_class).any():
        raise ValueError(f"Párrafos insuficientes para regenerar el corpus: {counts.to_dict()}")

    sources = corpus[SOURCE_COLUMNS[1:]].copy()
    sources.insert(0, "text_sha256", corpus["text"].map(text_hash))
    return corpus[DATASET_COLUMNS], sources


def read_dataset(csv_path, sources_path):
    """Lee el CSV del enunciado y comprueba la correspondencia con su procedencia."""
    def leer_csv(path):
        # Limitamos la espera para que una descarga no bloquee el cuaderno.
        if str(path).startswith("https://"):
            with urlopen(Request(path, headers={"User-Agent": USER_AGENT}), timeout=15) as response:
                path = BytesIO(response.read())
        return pd.read_csv(path, sep=";", encoding="utf-8", keep_default_na=False)

    data = leer_csv(csv_path)
    sources = leer_csv(sources_path)
    if list(data.columns) != DATASET_COLUMNS or list(sources.columns) != SOURCE_COLUMNS:
        raise ValueError("Las cabeceras del corpus o de su fichero de fuentes son incorrectas")
    if data.empty or data["text"].str.strip().eq("").any() or data["text"].duplicated().any():
        raise ValueError("El corpus está vacío o contiene textos vacíos o duplicados")
    if data["text"].str.contains(";", regex=False).any() or set(data["class"]) != set(CATEGORIAS):
        raise ValueError("El corpus contiene separadores o clases no válidas")
    # Comprobamos que cada texto coincide con su fila de procedencia.
    if not data["text"].map(text_hash).equals(sources["text_sha256"]):
        raise ValueError("El corpus y la procedencia no corresponden fila a fila")
    for column in ("article_id", "article_revision"):
        if not pd.api.types.is_integer_dtype(sources[column]) or (sources[column] <= 0).any():
            raise ValueError(f"Identificador inválido en {column}")
    if sources["article_title"].str.strip().eq("").any():
        raise ValueError("Hay artículos de origen sin título")
    return pd.concat([data, sources], axis=1)

### **Generación o carga del dataset**

Con `REGENERAR_DATASET = False` cargamos los CSV de partida del repositorio público, que conservan las revisiones utilizadas. Con `True` consultamos Wikipedia para los 62 artículos seleccionados. Si falla una fuente probamos la otra e indicamos el origen. Una regeneración puede cambiar los textos y resultados.

Comprobamos la procedencia, eliminamos residuos, conservamos la muestra histórica y no exportamos para publicación si faltan artículos. La huella permite comprobar que reproducimos el corpus evaluado.


In [ ]:
# Elegimos si regeneramos el corpus o utilizamos el ya publicado.
REGENERAR_DATASET = False


from sklearn.model_selection import StratifiedGroupKFold
import matplotlib.pyplot as plt


def crear_particiones(corpus):
    # Separamos por artículo para no evaluar párrafos de artículos ya vistos.
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    splits = list(splitter.split(corpus["text"], corpus["class"], corpus["article_id"]))
    for train, test in splits:
        if not set(corpus["article_id"].iloc[train]).isdisjoint(corpus["article_id"].iloc[test]):
            raise ValueError("Hay artículos compartidos entre entrenamiento y prueba.")
        if set(corpus["class"].iloc[train]) != set(LABELS):
            raise ValueError("Alguna partición de entrenamiento no contiene las tres clases.")
    return splits


def validar_corpus(corpus):
    # Comprobamos que tenemos suficientes artículos y párrafos para evaluar.
    articulos = corpus.groupby("class")["article_id"].nunique().reindex(LABELS, fill_value=0)
    counts = corpus["class"].value_counts().reindex(LABELS, fill_value=0)
    if (counts < 30).any():
        raise ValueError("Corpus demasiado pequeño para nuestra evaluación.")
    if (counts <= 300).any():
        print("Aviso: alguna clase no supera los 300 párrafos recomendados.")
    if (articulos < 5).any():
        raise ValueError("No hay suficientes artículos para la validación cruzada.")
    if corpus.groupby("article_id")["class"].nunique().gt(1).any():
        raise ValueError("Un artículo tiene más de una etiqueta.")
    # Comprobamos las particiones antes de aceptar el corpus; si fallan, probamos la otra fuente.
    crear_particiones(corpus)
    if corpus["text"].str.contains(r";|https?://|www\.|<[^>]+>", regex=True).any():
        raise ValueError("Hay marcas residuales en el corpus.")
    return corpus


def limpiar_corpus(corpus):
    # Aplicamos la misma limpieza a los datos publicados y a los recién obtenidos.
    corpus = corpus.copy()
    original = corpus["text"].copy()
    corpus["text"] = original.map(limpiar_texto)
    print(f"Párrafos con limpieza adicional: {(original != corpus['text']).sum()}")
    corpus = corpus[corpus["text"].str.len() >= 60].copy()
    # Si la limpieza crea duplicados, evitamos repetirlos o asignarles dos clases.
    clases = corpus.groupby("text")["class"].transform("nunique")
    corpus = corpus[clases == 1].drop_duplicates("text").reset_index(drop=True)
    corpus["text_sha256"] = corpus["text"].map(text_hash)
    return corpus


def cargar_publicado():
    # Leemos la pareja de CSV publicados y comprobamos que contiene los artículos previstos.
    corpus = read_dataset(f"{raw_url}/{DATASET_CSV}", f"{raw_url}/{SOURCES_CSV}")
    for clase, titulos in CATEGORIAS.items():
        if set(corpus.loc[corpus["class"] == clase, "article_title"]) != set(titulos):
            raise ValueError(f"Los CSV publicados no contienen los artículos previstos de {clase}.")
    return corpus


def regenerar_corpus():
    corpus, fuentes = collect_dataset(minimum_per_class=30)
    return corpus.join(fuentes)


def es_referencia_residual(texto):
    # Detectamos referencias con formato de cita, no por su temática.
    # Exigimos indicios bibliográficos concretos, incluido un año.
    fecha = re.search(r"\b(?:19|20)\d{2}\b", texto)
    blog = re.search(r"\bblogo\s+Scivolemo\b", texto, flags=re.IGNORECASE)
    libro = (re.search(r"\bEldonejo\b", texto, flags=re.IGNORECASE)
             and re.search(r"\b\d+\s+paĝoj\b", texto, flags=re.IGNORECASE))
    return texto.startswith(RESIDUOS_AUXILIARES) or bool(fecha and (blog or libro) and len(texto) < 230)


def obtener_corpus(regenerar=False):
    opciones = [("Wikipedia", regenerar_corpus), ("CSV publicados", cargar_publicado)]
    if not regenerar:
        opciones.reverse()
    for origen, cargar in opciones:
        try:
            corpus = cargar()
            total = len(corpus)
            partida = corpus
            previo = validar_corpus(limpiar_corpus(partida))
            # Conservamos la versión auditada y limpiamos las referencias detectadas después.
            referencias = previo["text"].map(es_referencia_residual)
            corpus = validar_corpus(previo.loc[~referencias].reset_index(drop=True))
            print(f"Origen del corpus: {origen}")
            print(f"Párrafos: {total} recibidos → {len(previo)} antes de la última limpieza → {len(corpus)} finales.")
            return partida, previo, corpus
        except Exception as error:
            # Si falla una fuente, probamos la otra.
            print(f"No se pudo obtener el corpus desde {origen}: {error}")
    print("No hay datos disponibles. Se omite la evaluación y la clasificación web.")
    return None, None, None


data_partida, data_previa, data = obtener_corpus(REGENERAR_DATASET)
if data is not None:
    # Exportamos solo si todos los artículos aportan texto al corpus final.
    faltantes = {titulo for titulos in CATEGORIAS.values() for titulo in titulos} - set(data["article_title"])
    if faltantes:
        print(f"AVISO: corpus parcial; faltan {sorted(faltantes)}. No exportamos CSV para publicar en esta ejecución. No uses archivos de una ejecución anterior.")
    else:
        for tabla, sufijo in ((data_partida, ""), (data, "_final")):
            tabla[DATASET_COLUMNS].to_csv(f"esperanto_scienco_dataset{sufijo}.csv", sep=";", encoding="utf-8", index=False)
            tabla[SOURCE_COLUMNS].to_csv(f"esperanto_scienco_fuentes{sufijo}.csv", sep=";", encoding="utf-8", index=False)
        print("Cuatro CSV exportados para publicar.")
    # Resumimos el tamaño del corpus y comprobamos el formato de salida.
    assert list(data[DATASET_COLUMNS].columns) == ["text", "class"]
    assert not data["text"].duplicated().any()
    assert data["text"].map(text_hash).equals(data["text_sha256"])
    assert not data["text"].str.contains(r";|https?://|www\.|<[^>]+>", regex=True).any()
    summary = data.groupby("class").agg(Párrafos=("text", "size"), Artículos=("article_id", "nunique"))
    display(summary.reindex(LABELS))
    # Visualizamos el equilibrio entre las tres categorías.
    ax = summary.reindex(LABELS)["Párrafos"].plot.bar(figsize=(5.8, 3.0), color="#4878a8", rot=0)
    ax.set(title="Distribución del corpus", xlabel="Categoría", ylabel="Párrafos")
    ax.bar_label(ax.containers[0], padding=3)
    ax.set_ylim(0, summary["Párrafos"].max() * 1.15)
    plt.tight_layout()
    plt.show()
    print(f"Corpus: {len(data)} párrafos y {data['article_id'].nunique()} artículos.")
    huella = text_hash("\n".join(sorted(data["text_sha256"] + ":" + data["class"])))
    print("Huella del corpus:", huella)
    if huella != "256553572f7961fd6ccb04113f82f97baaf8709129f3457eca792ca3c0465a4d":
        print("AVISO: el corpus difiere de la ampliación evaluada; las métricas pueden cambiar.")

# Recuperamos los artículos originales para la comparación controlada.
base_original = None
if data is not None:
    titulos_base = {titulo for titulos in ARTICULOS_ORIGINALES.values() for titulo in titulos}
    base_original = data[data["article_title"].isin(titulos_base)].reset_index(drop=True)
    huella_base = text_hash("\n".join(sorted(base_original["text_sha256"] + ":" + base_original["class"])))
    if huella_base != "6c1e9cf79d5c2ac2bf8189422b3660a6aa08d4c20c02ecca6689fa34bfcae802":
        print("Aviso: no se conserva la base histórica exacta; omitimos la comparación de ampliación.")
        base_original = None

### **Ruido de etiquetado**

Al heredar la categoría del artículo, un párrafo puede resultar genérico o tratar aspectos de otra disciplina. Buscamos raíces de términos científicos de las tres categorías y distinguimos cuatro casos: solo clase asignada, varias clases, solo otra clase y sin coincidencias.

El indicador nos ayuda a localizar posibles problemas, pero no determina si una etiqueta es correcta: el vocabulario puede ser compartido y nuestro diccionario no cubre todos los términos. Por eso contrastamos el análisis automático con una revisión de una muestra histórica. Primero analizamos el corpus final y después recuperamos los textos revisados antes de la última limpieza.


In [ ]:
# Definimos raíces orientativas de vocabulario temático en Esperanto.
# Buscamos prefijos de palabras completas para incluir variantes flexivas.
RAICES_TEMATICAS = {
    "biologio": ("biologi", "ĉel", "genetik", "genom", "organism", "plant",
                 "animal", "bakteri", "protein", "enzim", "metabol", "fotosintez",
                 "ekologi", "anatom"),
    "kemio": ("kemi", "ĥemi", "atom", "molekul", "jon", "acid", "baz", "reakci",
              "oksid", "kataliz", "solvaĵ", "element", "komponaĵ", "elektron", "alkohol"),
    "fiziko": ("fizik", "mekanik", "meĥanik", "gravita", "energi", "fort", "mov", "ond",
               "magnet", "kvant", "termodinamik", "optik", "lum", "rapid", "mas"),
}
ESTADOS_RUIDO = ["Solo clase asignada", "Varias clases", "Solo otra clase", "Sin coincidencias"]


def indicios_tematicos(texto, clase):
    # Contamos raíces distintas, no repeticiones de una misma palabra.
    palabras = set(re.findall(r"[^\W\d_]+", texto.lower()))
    coincidencias = {
        tema: [raiz for raiz in raices if any(p.startswith(raiz) for p in palabras)]
        for tema, raices in RAICES_TEMATICAS.items()
    }
    temas = [tema for tema, raices in coincidencias.items() if raices]
    if not temas:
        estado = "Sin coincidencias"
    elif len(temas) > 1:
        estado = "Varias clases"
    elif temas[0] == clase:
        estado = "Solo clase asignada"
    else:
        estado = "Solo otra clase"
    detalle = "; ".join(f"{tema}: {', '.join(coincidencias[tema])}" for tema in temas)
    return estado, detalle


if data is not None:
    # Aplicamos el mismo criterio a todos los párrafos, sin cambiar el dataset.
    ruido = data[["class", "article_title", "text"]].copy()
    ruido[["Indicio", "Raíces encontradas"]] = pd.DataFrame(
        [indicios_tematicos(texto, clase) for texto, clase in zip(ruido["text"], ruido["class"])],
        index=ruido.index)
    recuento = pd.crosstab(ruido["class"], ruido["Indicio"]).reindex(
        index=LABELS, columns=ESTADOS_RUIDO, fill_value=0)
    display(recuento)
    total = recuento.sum()
    display(pd.DataFrame({"Párrafos": total, "Porcentaje (%)": (100 * total / len(ruido)).round(1)}))
    # Mostramos un ejemplo aleatorio de cada señal, sin anotaciones precalculadas.
    candidatos = ruido[ruido["Indicio"] != "Solo clase asignada"]
    if not candidatos.empty:
        ejemplos = candidatos.groupby("Indicio", group_keys=False).sample(n=1, random_state=SEED)
        with pd.option_context("display.max_colwidth", None):
            display(ejemplos[["class", "article_title", "text", "Indicio", "Raíces encontradas"]]
                    .rename(columns={"class": "Clase heredada", "article_title": "Artículo", "text": "Párrafo"})
                    .reset_index(drop=True))
    print("Los recuentos son indicadores léxicos, no una tasa validada de etiquetas incorrectas.")


### **Revisión de una muestra**

Revisamos 24 párrafos del corpus original anterior a la última limpieza y a la sustitución de los dos artículos: elegimos tres al azar por grupo del indicador original y luego otros tres de los restantes, con semilla 42 en ambas selecciones. Cada integrante valoró los textos traducidos al español con ayuda de IA: **C** (claro), **A** (ambiguo) o **N** (no representativo). Al menos un integrante consultó también una valoración orientativa de IA. Registramos únicamente los votos propios del equipo. El acuerdo describe las coincidencias observadas y no garantiza que las valoraciones fueran independientes.

Calculamos la mayoría de los tres votos y el porcentaje de coincidencias. Esta muestra equilibrada entre grupos sirve para contrastar el indicador, no para estimar directamente el ruido de todo el corpus.

Guardamos las valoraciones de esa muestra, sin volver a seleccionar textos en cada ejecución. Si Wikipedia cambia, analizamos solo los párrafos originales que sigan disponibles. El cruce usa el indicador actualizado, que incluye las variantes ortográficas ĥemi y meĥanik.

La ampliación no cambia las valoraciones originales. Los 24 textos forman una muestra histórica del corpus anterior; no los interpretamos como una muestra representativa de los artículos nuevos.


In [ ]:
# Registramos los votos de Nicolás, Gonzalo y Hugo, en ese orden.
# La huella identifica el párrafo revisado aunque cambie su posición en el CSV.
REVISION_MANUAL = [
    (1, "16ae4ec6532bdc8e571adb2a584cd71ab03cd5bae723686699830ada2cf3b87a", "CCC"),
    (2, "3dca5aeabda7fdce72f333ab4de058075c1530e5400ee0c1a7d61ebd0ce335e9", "CCA"),
    (3, "c02af64fd6c84d8b888b48499196fdafac3dcf22856f1c6887477565da07fa01", "CCC"),
    (4, "02217058b83c7044f3bc6fc75914db7b4b8d42a94aebde8a47977a5747e46fbd", "NNN"),
    (5, "b6ec440e04f7a7d10b25dd98756b6c76e4cd64ba240ce270592a75e391bd552a", "CCC"),
    (6, "81faac89f9fe457758ffa7d0f0a28bef71865a463b329627d527674bdc8dcf67", "CCC"),
    (7, "b4126fb816c071590150d976f6e7782502f3bb653a7aec99877db698eb1ac974", "CCC"),
    (8, "81884f2ade081c91f6626bbcbb3a02756d2ecc046ed04587a0acc5d727637926", "CAA"),
    (9, "e1f6564bfd444b664f9fa894a40bfd7c7e096f1fd3899acf62e00af69f889a33", "CCC"),
    (10, "5e658bd3db8077c25b67e140f212c08106a607d3000729a1fe565e809752b34a", "CCC"),
    (11, "dcfd32fef4997c0baaa111f52fae06c6698947c2c4a0720d6e7f9c14f005d5e9", "CCC"),
    (12, "2cda117556fbcaa3023e66960ce23eeeaa33eab54694c40ddf6a6f7577c6ea91", "AAA"),
    (13, "661794736d080e827005b3a3529f1a22380649fb04c48ce0f815d8d09ff9ea2c", "CCC"),
    (14, "6eef1be00d3fa7a80b741d13c4b5eaebc05ba105569a6d608fdb8486d5d82166", "CCC"),
    (15, "b6e45c5b26a868a0fefad890daf4bef3fbcc292b3b4f55129d47e3e8dd8d704a", "CCC"),
    (16, "1c810faee2c07beaa84a7e6c5cdcc02792c3aa88054f72e54c7a26cf5689ff8c", "AAA"),
    (17, "a78cbe974595f5269f7ac74703078ad7e1219f8d9611a17e8780876681da9c93", "ANN"),
    (18, "ea513c4749e2382ba7d8b8e4c4837262f6212666871de2fcf960533fcae587be", "CCC"),
    (19, "bcd0bf7e9749e60d3cc071f06a960d4de7aea8a21a763a354f1c96b4aaaad243", "NNN"),
    (20, "fbae2d0da84b73c3a734b913339387178e07e0b4b3a9c1163af7ad1ad932dc3e", "CCC"),
    (21, "6c69196c65ba72a610120abc7b333bf27d685cf2eedda33193d90551aac815da", "CCC"),
    (22, "6ed8107eac28616b067888dbff61fdc715de37f23ac8059bd8a7d8a652f17b78", "CCC"),
    (23, "5278488b778e0680e10035e0d092c8c0bafd6a6f4ad124040fa49d30e82a6b5e", "CCC"),
    (24, "25f75c3a010f40de7694d8a70bec5d957db35347ee42b0d5621f03f57dc8858a", "CCC"),
]


In [ ]:
# Recuperamos los textos de la muestra y sus tres valoraciones.
if data_previa is not None:
    votos = pd.DataFrame(REVISION_MANUAL, columns=["N.º", "text_sha256", "Votos"])
    revision = votos.merge(data_previa, on="text_sha256", how="inner", validate="one_to_one")
    print(f"Párrafos revisados disponibles: {len(revision)}/{len(votos)}")
    if len(revision) != len(votos):
        print("El corpus ha cambiado; analizamos solo los textos que coinciden.")

    if not revision.empty:
        revisores = ["Nicolás", "Gonzalo", "Hugo"]
        revision[revisores] = pd.DataFrame(revision["Votos"].map(list).tolist(), index=revision.index)
        nombres = {"C": "Claro", "A": "Ambiguo", "N": "No representativo"}

        # Dos votos iguales bastan para establecer la mayoría.
        def mayoria(votos):
            for valor in votos:
                if votos.count(valor) >= 2:
                    return nombres[valor]
            return "Sin mayoría"

        revision["Mayoría"] = revision["Votos"].map(mayoria)
        revision["Indicio"] = [indicios_tematicos(t, c)[0]
                               for t, c in zip(revision["text"], revision["class"])]
        revision["Conservado"] = revision["text_sha256"].isin(data["text_sha256"])
        display(revision[["N.º", "article_title", "class", "Indicio"] + revisores + ["Mayoría", "Conservado"]]
                .rename(columns={"article_title": "Artículo", "class": "Etiqueta heredada"}))

        # Cruzamos las valoraciones humanas con el indicador automático.
        tabla = pd.crosstab(revision["Indicio"], revision["Mayoría"])
        columnas = list(nombres.values())
        if "Sin mayoría" in tabla.columns:
            columnas.append("Sin mayoría")
        tabla = tabla.reindex(index=ESTADOS_RUIDO, columns=columnas, fill_value=0)
        tabla["Total"] = tabla.sum(axis=1)
        tabla.loc["Total"] = tabla.sum()
        display(tabla)

        # Resumimos los resultados de los textos recuperados, sin fijar cifras a mano.
        for valor in columnas:
            cantidad = int(tabla.loc["Total", valor])
            print(f"{valor}: {cantidad}/{len(revision)} ({cantidad / len(revision):.1%})")
        if len(revision) == len(votos) and tabla.loc[ESTADOS_RUIDO, "Total"].eq(6).all():
            print("Muestra completa: seis párrafos por grupo del indicador.")
            print(f"Claros sin coincidencias: {tabla.loc['Sin coincidencias', 'Claro']}/6.")
        else:
            print("El cruce corresponde a los textos recuperados, no necesariamente a toda la muestra original.")

        # Medimos coincidencias entre los tres y entre cada pareja.
        unanimidad = revision[revisores].nunique(axis=1).eq(1).mean()
        acuerdos = pd.Series({
            "Nicolás y Gonzalo": revision["Nicolás"].eq(revision["Gonzalo"]).mean(),
            "Nicolás y Hugo": revision["Nicolás"].eq(revision["Hugo"]).mean(),
            "Gonzalo y Hugo": revision["Gonzalo"].eq(revision["Hugo"]).mean(),
        })
        print(f"Unanimidad: {unanimidad:.1%}")
        print(f"Acuerdo medio por parejas: {acuerdos.mean():.1%}")
        display((100 * acuerdos).round(1).rename("Acuerdo (%)").to_frame())

    # Mostramos qué referencias se descartaron tras la auditoría.
    descartados = data_previa[~data_previa["text_sha256"].isin(data["text_sha256"])]
    print(f"Referencias descartadas: {len(descartados)}. Corpus final: {len(data)} párrafos.")
    if not descartados.empty:
        with pd.option_context("display.max_colwidth", None):
            display(descartados[["article_title", "class", "text"]].rename(
                columns={"article_title": "Artículo", "class": "Etiqueta", "text": "Referencia"}))


#### **Interpretación de la revisión**

El cruce anterior nos permite comprobar si las señales del diccionario coinciden con nuestras valoraciones. Encontrar términos de varias disciplinas no implica necesariamente que un párrafo sea ambiguo. Tampoco podemos considerar incorrecto un texto solo porque no aparezca ninguna de nuestras raíces.

En la revisión original nos llamaron la atención tres casos:

- **16 (Atomo):** lo valoramos como ambiguo. Predomina la física cuántica, aunque el artículo pertenece a química en nuestro dataset.
- **18 (Optiko):** los tres consideramos que las ilusiones ópticas encajan en física.
- **19 (Genetiko):** encontramos una referencia bibliográfica, no un párrafo científico.

A partir del último caso ampliamos la limpieza para retirar referencias de blog, libros con datos editoriales y otros textos auxiliares. Conservamos los párrafos científicos ambiguos y sus etiquetas originales. Los resultados de la revisión corresponden a la muestra anterior a esta limpieza.

## **2. NLP en Esperanto**

Construimos un NLP basado en reglas, sin utilizar librerías lingüísticas preexistentes. Separamos palabras y signos con una expresión regular. Consultamos primero el diccionario de palabras funcionales y después aplicamos las terminaciones gramaticales. Por ejemplo, `ĉelojn` (plural y acusativo) se convierte en `ĉelo`.

Cada `Token` guarda el texto, el lema, la categoría gramatical y las marcas de stopword y puntuación. `Doc` agrupa los tokens y permite recorrerlos y recuperar sus lemas. Nuestra interfaz se inspira en spaCy. No hacemos análisis sintáctico ni resolvemos todas las ambigüedades dependientes del contexto.

Como referencia para contrastar las terminaciones del Esperanto, la guía de la práctica recomienda su [guía gramatical](https://jparisu.github.io/nlp-esperantilo/es/guide/esperanto/).


In [ ]:
"""Análisis lingüístico de Esperanto mediante reglas gramaticales."""

import re


ESPERANTO_STOPWORDS = {
    "la", "kaj", "de", "en", "al", "estas", "esti",
    "estis", "estos", "por", "kun", "per", "sur",
    "pri", "sed", "ĉar", "ke", "ne", "kiel", "tiu",
    "tiuj", "tiun", "tiujn", "kiu", "kiuj", "kiun",
    "kiujn", "mi", "vi", "li", "ŝi", "ĝi", "ili",
    "ni", "oni", "sia", "sian", "siaj", "siajn",
    "ankaŭ", "nur", "jam", "pli", "plej", "tre",
    "tro", "ĉe", "post", "dum"
}

# Definimos las palabras frecuentes que no podemos clasificar solo por su terminación.

PALABRAS_FUNCION = {
    "la": ("la", "DET"), "hodiaŭ": ("hodiaŭ", "ADV"),
    "hieraŭ": ("hieraŭ", "ADV"), "morgaŭ": ("morgaŭ", "ADV"),
    "nun": ("nun", "ADV"), "pli": ("pli", "ADV"),
    "plej": ("plej", "ADV"), "tre": ("tre", "ADV"),
    "tro": ("tro", "ADV"), "kaj": ("kaj", "CCONJ"),
    "sed": ("sed", "CCONJ"), "ĉar": ("ĉar", "SCONJ"),
    "ke": ("ke", "SCONJ"), "de": ("de", "ADP"),
    "en": ("en", "ADP"), "al": ("al", "ADP"),
    "por": ("por", "ADP"), "kun": ("kun", "ADP"),
    "per": ("per", "ADP"), "sur": ("sur", "ADP"),
    "pri": ("pri", "ADP"), "ĉe": ("ĉe", "ADP"),
    "post": ("post", "ADP"), "dum": ("dum", "ADP"),
    "ne": ("ne", "PART"), "kiel": ("kiel", "ADV"),
    "jam": ("jam", "ADV"), "ankaŭ": ("ankaŭ", "ADV"),
    "nur": ("nur", "ADV"), "mi": ("mi", "PRON"),
    "vi": ("vi", "PRON"), "li": ("li", "PRON"),
    "ŝi": ("ŝi", "PRON"), "ĝi": ("ĝi", "PRON"),
    "ili": ("ili", "PRON"), "ni": ("ni", "PRON"),
    "oni": ("oni", "PRON"), "tiu": ("tiu", "DET"),
    "tiuj": ("tiu", "DET"), "tiun": ("tiu", "DET"),
    "tiujn": ("tiu", "DET"), "kiu": ("kiu", "PRON"),
    "kiuj": ("kiu", "PRON"), "kiun": ("kiu", "PRON"),
    "kiujn": ("kiu", "PRON"), "ĉiu": ("ĉiu", "PRON"),
    "neniu": ("neniu", "PRON"), "iu": ("iu", "PRON"),
    "kio": ("kio", "PRON"), "tio": ("tio", "PRON"),
    "io": ("io", "PRON"), "ĉio": ("ĉio", "PRON"),
    "nenio": ("nenio", "PRON"), "kiam": ("kiam", "ADV"),
    "tiam": ("tiam", "ADV"), "kie": ("kie", "ADV"),
    "tie": ("tie", "ADV"), "kial": ("kial", "ADV"),
    "tial": ("tial", "ADV"), "se": ("se", "SCONJ"),
    "kvankam": ("kvankam", "SCONJ"), "aŭ": ("aŭ", "CCONJ"),
    "nek": ("nek", "CCONJ"), "sen": ("sen", "ADP"),
    "el": ("el", "ADP"), "ĝis": ("ĝis", "ADP"),
    "inter": ("inter", "ADP"), "tra": ("tra", "ADP"),
    "kontraŭ": ("kontraŭ", "ADP"), "antaŭ": ("antaŭ", "ADP"),
    "sub": ("sub", "ADP"), "super": ("super", "ADP"),
    "si": ("si", "PRON"), "jes": ("jes", "PART"),
    "ĉu": ("ĉu", "PART"), "do": ("do", "ADV"),
    "ol": ("ol", "SCONJ"), "malgraŭ": ("malgraŭ", "ADP"),
    "pro": ("pro", "ADP"), "da": ("da", "ADP"),
    "je": ("je", "ADP"), "po": ("po", "ADP"),
    "laŭ": ("laŭ", "ADP"), "ĉirkaŭ": ("ĉirkaŭ", "ADP"),
    "anstataŭ": ("anstataŭ", "ADP"), "plu": ("plu", "ADV"),
    "tiel": ("tiel", "ADV"), "tamen": ("tamen", "ADV"),
    "ĉiam": ("ĉiam", "ADV"), "neniam": ("neniam", "ADV"),
    "tiom": ("tiom", "ADV"), "iom": ("iom", "ADV"),
    "ankoraŭ": ("ankoraŭ", "ADV"), "preskaŭ": ("preskaŭ", "ADV"),
    "almenaŭ": ("almenaŭ", "ADV"), "eĉ": ("eĉ", "PART"),
    "ĉi": ("ĉi", "PART"), "ajn": ("ajn", "PART"), "mem": ("mem", "PART"),
    "ties": ("ties", "DET"), "kies": ("kies", "DET"), "ambaŭ": ("ambaŭ", "DET"),
}

# Normalizamos los pronombres personales en acusativo.
for pronombre in ("mi", "vi", "li", "ŝi", "ĝi", "ni", "ili", "oni", "si"):
    PALABRAS_FUNCION[pronombre + "n"] = (pronombre, "PRON")
# Tratamos el plural y el acusativo de los correlativos.
for palabra in ("ĉiu", "neniu", "iu"):
    for sufijo in ("j", "n", "jn"):
        PALABRAS_FUNCION[palabra + sufijo] = (palabra, "PRON")
for palabra in ("kio", "tio", "io", "ĉio", "nenio"):
    PALABRAS_FUNCION[palabra + "n"] = (palabra, "PRON")

# Añadimos palabras funcionales frecuentes a las stopwords.
ESPERANTO_STOPWORDS.update({"si", "se", "aŭ", "nek", "sen", "el", "ĝis",
                           "inter", "tra", "kontraŭ", "antaŭ", "sub", "super"})

# Incluimos los posesivos y sus formas de plural y acusativo.
for posesivo in ("mia", "via", "lia", "ŝia", "ĝia", "nia", "ilia", "sia"):
    for sufijo in ("", "j", "n", "jn"):
        PALABRAS_FUNCION[posesivo + sufijo] = (posesivo, "DET")
    ESPERANTO_STOPWORDS.add(posesivo)

ESPERANTO_STOPWORDS.update({"pro", "da", "je", "po", "laŭ", "ĉirkaŭ", "anstataŭ",
                           "plu", "tiel", "tamen", "ĉiam", "neniam", "ĉi", "ties", "kies",
                           "eĉ", "ambaŭ", "mem", "ajn", "tiom", "iom", "ankoraŭ",
                           "preskaŭ", "almenaŭ", "ĉiu", "tio", "kio", "kiam"})

# Completamos las palabras funcionales y los correlativos frecuentes.
for pos, palabras in {
    "ADP": ("krom", "apud", "ekster", "trans"),
    "ADV": ("ju", "des", "tuj", "kiom", "iam", "ial", "ĉiel"),
    "PART": ("jen", "ja"),
    "SCONJ": ("kvazaŭ",),
    "DET": ("ĉies",),
}.items():
    for palabra in palabras:
        PALABRAS_FUNCION[palabra] = (palabra, pos)

# Completamos la tabla de correlativos y los tratamos todos como stopwords.
for prefijo in ("ki", "ti", "i", "ĉi", "neni"):
    for sufijo, pos in (("al", "ADV"), ("am", "ADV"), ("e", "ADV"),
                        ("el", "ADV"), ("om", "ADV"), ("es", "DET")):
        PALABRAS_FUNCION.setdefault(prefijo + sufijo, (prefijo + sufijo, pos))
        ESPERANTO_STOPWORDS.add(prefijo + sufijo)

# Usamos las clases cerradas como criterio adicional de stopwords.
CLASES_CERRADAS = {"ADP", "DET", "PRON", "CCONJ", "SCONJ", "PART"}
ESPERANTO_STOPWORDS.update(lema for lema, pos in PALABRAS_FUNCION.values()
                           if pos in CLASES_CERRADAS)

# Añadimos los numerales básicos que no siguen las terminaciones habituales.
NUMERALES = {"nul", "unu", "du", "tri", "kvar", "kvin", "ses", "sep",
            "ok", "naŭ", "dek", "cent", "mil"}


class Token:
    """Representa un token con su lema y propiedades gramaticales."""

    def __init__(self, text, lemma, pos, is_stop, is_punct):
        self.text = text
        self.lemma_ = lemma
        self.pos_ = pos
        self.is_stop = is_stop
        self.is_punct = is_punct

    def __repr__(self):
        return (
            f"Token(text={self.text!r}, lemma={self.lemma_!r}, "
            f"pos={self.pos_!r}, stop={self.is_stop}, "
            f"punct={self.is_punct})"
        )


class Doc:
    """Agrupa los tokens y permite recorrerlos o recuperar sus lemas."""

    def __init__(self, tokens, original_text):
        self._tokens = tokens
        self._text = original_text

    def __iter__(self):
        return iter(self._tokens)

    def __len__(self):
        return len(self._tokens)

    def __getitem__(self, index):
        return self._tokens[index]

    @property
    def text(self):
        return self._text

    def lemmatized_text(self, remove_stopwords=True, remove_punct=True):
        # Recorremos los tokens y conservamos los lemas que nos interesan.
        lemmas = []
        for token in self._tokens:

            if remove_punct and token.is_punct:
                continue

            if remove_stopwords and token.is_stop:
                continue

            lemmas.append(token.lemma_)
        return " ".join(lemmas)


class EsperantoNLP:
    """Modelo de NLP basado en reglas para Esperanto."""

    def __init__(self, stopwords=None):
        self.stopwords = (
            set(ESPERANTO_STOPWORDS)
            if stopwords is None
            else set(stopwords)
        )

    def _morphological_analysis(self, word):
        w = word.lower()

        # Consultamos las palabras funcionales antes de aplicar los sufijos.
        if w in PALABRAS_FUNCION:
            return PALABRAS_FUNCION[w]
        # Reconocemos ordinales escritos con cifras, incluidos plural y acusativo.
        ordinal = re.fullmatch(r"(\d+)-a(?:jn|j|n)?", w)
        if ordinal:
            return ordinal[1] + "-a", "ADJ"
        # Tratamos enteros y decimales como números, y las fórmulas con cifras como símbolos.
        if re.fullmatch(r"\d+(?:[.,]\d+)*", w) or w in NUMERALES:
            return w, "NUM"
        if any(caracter.isdigit() for caracter in w):
            return w, "X"

        # No interpretamos las terminaciones de las siglas en mayúsculas.
        if word.isalpha() and word.isupper() and len(word) > 1:
            return w, "X"

        # Reconocemos el acusativo de dirección de los adverbios (hejmen, supren, kien).
        if len(w) > 2 and w.endswith("en") and w[:-1].endswith("e"):
            return w[:-1], "ADV"

        # Reconocemos los verbos y los reducimos al infinitivo.
        for ending in ("as", "is", "os", "us"):

            if w.endswith(ending) and len(w) > len(ending):
                return w[:-len(ending)] + "i", "VERB"
        if w.endswith("u") and len(w) > 2:
            return w[:-1] + "i", "VERB"
        if w.endswith("i") and len(w) > 2:
            return w, "VERB"

        # Quitamos marcas de plural (-j) y acusativo (-n) cuando procede.
        base = w
        if (
            len(w) > 3
            and w.endswith("jn")
            and w[:-2].endswith(("o", "a"))
        ):
            base = w[:-2]
        elif (
            len(w) > 2
            and w.endswith("n")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]
        elif (
            len(w) > 2
            and w.endswith("j")
            and w[:-1].endswith(("o", "a"))
        ):
            base = w[:-1]

        # Usamos la última letra para reconocer sustantivos, adjetivos y adverbios.
        if base.endswith("o") and len(base) > 1:
            return base, "NOUN"
        if base.endswith("a") and len(base) > 1:
            return base, "ADJ"
        if base.endswith("e") and len(base) > 1:
            return base, "ADV"
        return w, "X"

    def __call__(self, text):
        if not isinstance(text, str):
            raise TypeError("EsperantoNLP espera una cadena de texto.")

        # Separamos palabras y signos de puntuación con una expresión regular.
        raw_tokens = re.findall(r"\d+-a(?:jn|j|n)?\b|\d+(?:[.,]\d+)+|\w+|[^\w\s]", text)

        # Construimos un Token por cada elemento y los agrupamos en un Doc.
        doc_tokens = []
        for raw in raw_tokens:

            is_punct = bool(re.fullmatch(r"[^\w\s]", raw))

            lower_raw = raw.lower()

            if is_punct:
                lemma, pos = raw, "PUNCT"
            else:
                lemma, pos = self._morphological_analysis(raw)

            token = Token(
                text=raw,
                lemma=lemma,
                pos=pos,
                is_stop=lower_raw in self.stopwords or lemma in self.stopwords,
                is_punct=is_punct
            )

            doc_tokens.append(token)
        return Doc(doc_tokens, text)

In [ ]:
# Contamos las comprobaciones superadas para no escribir el total a mano.
comprobaciones = 0


def comprobar(condicion):
    global comprobaciones
    assert condicion
    comprobaciones += 1


# Probamos el modelo con una frase y mostramos los atributos de cada token.
nlp = EsperantoNLP()
doc = nlp("La grandaj ĉeloj dividiĝas rapide en la laboratorio.")
display(pd.DataFrame([
    {"Token": token.text, "Lema": token.lemma_, "POS": token.pos_,
     "Stopword": token.is_stop, "Puntuación": token.is_punct}
    for token in doc
]))
print("Entrada del vectorizador:", doc.lemmatized_text())

# Comprobamos las reglas principales y la API con ejemplos sencillos.
comprobar(nlp("ĉelojn")[0].lemma_ == "ĉelo")
comprobar(nlp("grandajn")[0].pos_ == "ADJ")
comprobar(nlp("legis")[0].lemma_ == "legi")
comprobar(nlp("kaj")[0].is_stop)
comprobar(doc[-1].is_punct)
comprobar(nlp("du")[0].pos_ == "NUM")
comprobar(nlp("42")[0].pos_ == "NUM")
comprobar(nlp("hodiaŭ")[0].pos_ == "ADV")
comprobar(nlp("libroj")[0].lemma_ == "libro")
comprobar(nlp("rapide")[0].pos_ == "ADV")
comprobar(nlp("ĉelon")[0].lemma_ == "ĉelo")
comprobar(nlp("legos")[0].lemma_ == "legi")
comprobar(nlp("bela")[0].pos_ == "ADJ")
comprobar(doc.text == "La grandaj ĉeloj dividiĝas rapide en la laboratorio.")
comprobar(doc.lemmatized_text() == "granda ĉelo dividiĝi rapide laboratorio")
# Comprobamos las excepciones.
for palabra, lema, pos in [
    ("ĉiu", "ĉiu", "PRON"), ("neniu", "neniu", "PRON"),
    ("kio", "kio", "PRON"), ("kiam", "kiam", "ADV"),
    ("se", "se", "SCONJ"), ("aŭ", "aŭ", "CCONJ"),
    ("min", "mi", "PRON"), ("sen", "sen", "ADP"),
    ("ĉiujn", "ĉiu", "PRON"), ("ĉu", "ĉu", "PART"),
    ("ĝis", "ĝis", "ADP"), ("si", "si", "PRON"),
]:
    token = nlp(palabra)[0]
    comprobar((token.lemma_, token.pos_) == (lema, pos))
comprobar(nlp("min")[0].is_stop)
# Comprobamos las nuevas palabras funcionales, los posesivos y las siglas.
casos_nuevos = [
    ("pro", "pro", "ADP", True), ("da", "da", "ADP", True),
    ("je", "je", "ADP", True), ("po", "po", "ADP", True),
    ("plu", "plu", "ADV", True), ("laŭ", "laŭ", "ADP", True),
    ("ĉiam", "ĉiam", "ADV", True), ("anstataŭ", "anstataŭ", "ADP", True),
    ("iliajn", "ilia", "DET", True), ("ĝian", "ĝia", "DET", True),
    ("ĉiujn", "ĉiu", "PRON", True), ("DNA", "dna", "X", False),
]
for palabra, lema, pos, stop in casos_nuevos:
    token = nlp(palabra)[0]
    comprobar((token.lemma_, token.pos_, token.is_stop) == (lema, pos, stop))
casos_adicionales = [
    ("krom", "ADP"), ("apud", "ADP"), ("ekster", "ADP"), ("trans", "ADP"),
    ("ju", "ADV"), ("des", "ADV"), ("tuj", "ADV"), ("jen", "PART"),
    ("kvazaŭ", "SCONJ"), ("kiom", "ADV"), ("iam", "ADV"), ("ial", "ADV"),
    ("ĉiel", "ADV"), ("ĉies", "DET"), ("ja", "PART"),
]
for palabra, pos in casos_adicionales:
    comprobar(nlp(palabra)[0].pos_ == pos)
for palabra in ("ol", "kvankam", "ĉu", "malgraŭ", "iu", "neniu", "ĉio"):
    comprobar(nlp(palabra)[0].is_stop)
comprobar(len(nlp("20-a")) == 1 and nlp("20-a")[0].pos_ == "ADJ")
comprobar(nlp("20-ajn")[0].lemma_ == "20-a")
# Comprobamos correlativos, acusativo de dirección, decimales y fórmulas.
for palabra, lema, pos in [
    ("nenies", "nenies", "DET"), ("ĉial", "ĉial", "ADV"), ("neniel", "neniel", "ADV"),
    ("kien", "kie", "ADV"), ("hejmen", "hejme", "ADV"), ("H2O", "h2o", "X"),
    ("0,25", "0,25", "NUM"),
]:
    tokens = nlp(palabra)
    comprobar(len(tokens) == 1 and (tokens[0].lemma_, tokens[0].pos_) == (lema, pos))
print(f"{comprobaciones} comprobaciones del NLP superadas.")

## **3. Clasificación de textos**

Representamos los párrafos mediante TF-IDF, con palabras individuales y pares consecutivos (unigramas y bigramas) presentes en al menos dos párrafos del conjunto de entrenamiento. Comparamos primero la regresión logística con y sin nuestro NLP; después comparamos la regresión logística con Naive Bayes multinomial usando el mismo preprocesamiento.

Usamos cinco particiones de validación cruzada agrupadas por artículo para evitar que textos de un mismo artículo aparezcan a la vez en entrenamiento y prueba. Ajustamos TF-IDF dentro de cada partición. Evaluamos con accuracy, F1 macro y matriz de confusión. Elegimos el clasificador con NLP que obtiene mayor F1 macro y lo entrenamos con todo el corpus para clasificar nuevas páginas.

In [ ]:
"""Vectorización y clasificación de textos científicos en Esperanto."""

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from functools import lru_cache


_NLP = EsperantoNLP()


# Guardamos temporalmente los lemas para no repetir el mismo preprocesamiento.
@lru_cache(maxsize=4096)
def lemmatize(text):
    """Obtiene los lemas del texto antes de aplicar TF-IDF."""
    return _NLP(text).lemmatized_text()


def build_classifier(use_nlp=True, seed=42, naive_bayes=False):
    """Prepara el vectorizador y el clasificador, con o sin NLP propio."""
    # Elegimos entre los textos originales y los lemas de nuestro NLP.
    representation = (
        {"preprocessor": lemmatize, "tokenizer": str.split, "token_pattern": None}
        if use_nlp else {}
    )
    # Encadenamos la vectorización y el entrenamiento en un único pipeline.
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, **representation)),
        ("classifier", MultinomialNB() if naive_bayes else LogisticRegression(max_iter=1000, random_state=seed)),
    ])

### **Experimento controlado: ¿ayuda añadir artículos?**

Antes de evaluar el corpus ampliado, comparamos Naive Bayes con sus parámetros originales en las mismas cinco particiones históricas. En cada partición añadimos los artículos nuevos solo a entrenamiento y mantenemos exactamente el mismo test. Así evitamos atribuir una mejora a un cambio de los artículos evaluados. Mostramos también la validación agrupada de todo el corpus ampliado, que responde a una pregunta diferente. Los resultados son exploratorios, no una prueba independiente definitiva.

In [ ]:
# Comparamos el mismo Naive Bayes (alpha=1) sobre exactamente los mismos artículos de prueba.
# Los artículos nuevos se usan solo como entrenamiento en esta comparación.
if data is not None and base_original is not None:
    import numpy as np
    from sklearn.metrics import accuracy_score, f1_score, classification_report
    from sklearn.model_selection import StratifiedGroupKFold
    nuevos = data.loc[~data["article_id"].isin(base_original["article_id"])].copy()
    if nuevos.empty:
        print("No se añadieron artículos; no hay una ampliación que comparar.")
    else:
        y = base_original["class"]
        pred_base = np.empty(len(y), dtype=object)
        pred_ampliado = np.empty(len(y), dtype=object)
        filas = []
        for numero, (train, test) in enumerate(crear_particiones(base_original), start=1):
            entrenamiento = base_original.iloc[train]
            prueba = base_original.iloc[test]
            entrenamiento_ampliado = pd.concat([entrenamiento, nuevos], ignore_index=True)
            assert set(entrenamiento_ampliado["article_id"]).isdisjoint(prueba["article_id"])
            assert set(entrenamiento_ampliado["text_sha256"]).isdisjoint(prueba["text_sha256"])
            for nombre, datos, pred in (("Original", entrenamiento, pred_base),
                                        ("Ampliado", entrenamiento_ampliado, pred_ampliado)):
                m = build_classifier(naive_bayes=True).fit(datos["text"], datos["class"])
                pred[test] = m.predict(prueba["text"])
                filas.append({"Partición": numero, "Corpus de train": nombre,
                    "Párrafos train": len(datos), "Párrafos test": len(prueba),
                    "Train": accuracy_score(datos["class"], m.predict(datos["text"])),
                    "Test": accuracy_score(prueba["class"], pred[test])})
        display(pd.DataFrame(filas).round(3))
        comparacion = pd.DataFrame([
            {"Corpus": nombre, "Accuracy test": accuracy_score(y, pred),
             "F1 macro": f1_score(y, pred, labels=LABELS, average="macro")}
            for nombre, pred in (("Original", pred_base), ("Ampliado", pred_ampliado))])
        display(comparacion.round(4))
        por_clase = pd.concat({nombre: pd.DataFrame(classification_report(y, pred, labels=LABELS,
                    output_dict=True, zero_division=0)).T.loc[LABELS, ["precision", "recall", "f1-score"]]
                    for nombre, pred in (("Original", pred_base), ("Ampliado", pred_ampliado))}, axis=1)
        display(por_clase.round(3))
        print("Esta comparación usa solo los artículos originales como test. La validación completa de abajo evalúa también los artículos nuevos.")


In [ ]:
model = None
if data is not None:
    import matplotlib.pyplot as plt
    from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score
    import numpy as np

    # Medimos el tiempo de evaluación y entrenamiento.
    inicio = time.perf_counter()
    texts, labels, groups = data["text"], data["class"], data["article_id"]
    # Reutilizamos el mismo criterio de partición ya comprobado al cargar los datos.
    splits = crear_particiones(data)

    print(f"Validación: {len(splits)} particiones sin artículos compartidos.")
    display(pd.DataFrame([{"Partición": i, "Párrafos train": len(train), "Párrafos test": len(test),
                           "Artículos train": groups.iloc[train].nunique(),
                           "Artículos test": groups.iloc[test].nunique()}
                          for i, (train, test) in enumerate(splits, start=1)]))
    # Evaluamos las tres configuraciones sobre las mismas particiones.
    scores = []
    configuraciones = [("TF-IDF directo + LR", False, False),
                       ("TF-IDF + NLP + LR", True, False),
                       ("TF-IDF + NLP + NB", True, True)]
    predicciones = []
    probabilidades = []
    resultados_particiones = []
    detalle_train_test = []
    for name, use_nlp, naive_bayes in configuraciones:
        pipeline = build_classifier(use_nlp=use_nlp, seed=SEED, naive_bayes=naive_bayes)
        predictions = np.empty(len(labels), dtype=object)
        probabilities = np.zeros((len(labels), len(LABELS)))
        train_scores = []
        for numero, (train, test) in enumerate(splits, start=1):
            pipeline.fit(texts.iloc[train], labels.iloc[train])
            predictions[test] = pipeline.predict(texts.iloc[test])
            # Guardamos también las probabilidades de test, ordenadas como LABELS.
            columnas = [list(pipeline.classes_).index(label) for label in LABELS]
            probabilities[test] = pipeline.predict_proba(texts.iloc[test])[:, columnas]
            acc_train = accuracy_score(labels.iloc[train], pipeline.predict(texts.iloc[train]))
            train_scores.append(acc_train)
            detalle_train_test.append({"Modelo": name, "Partición": numero,
                "Train": acc_train, "Test": accuracy_score(labels.iloc[test], predictions[test])})
        scores.append({
            "Representación": name,
            "Train medio": np.mean(train_scores),
            "Accuracy": accuracy_score(labels, predictions),
            "F1 macro": f1_score(labels, predictions, average="macro"),
        })
        predicciones.append(predictions)
        probabilidades.append(probabilities)
        resultados_particiones.append([accuracy_score(labels.iloc[test], predictions[test])
                                       for _, test in splits])

    display(pd.DataFrame(scores).round(3))
    display(pd.DataFrame(detalle_train_test).round(3))
    # La diferencia train-test orienta sobre generalización, no demuestra por sí sola sobreajuste.
    resumen_train_test = pd.DataFrame(detalle_train_test).groupby("Modelo", sort=False)[["Train", "Test"]].mean()
    display(resumen_train_test.round(3))
    display(pd.DataFrame(detalle_train_test).groupby("Modelo", sort=False)[["Train", "Test"]].std(ddof=0).round(3))
    display(pd.DataFrame(resultados_particiones, index=[c[0] for c in configuraciones],
                         columns=[f"Partición {i+1}" for i in range(len(splits))]).T.round(3))
    # Elegimos entre los modelos con NLP usando F1 macro.
    mejor = max((1, 2), key=lambda i: scores[i]["F1 macro"])
    nlp_predictions = predicciones[mejor]
    nlp_probabilities = probabilidades[mejor]
    print("Clasificador seleccionado:", configuraciones[mejor][0])
    # La comparación anterior sirve para elegir; estimamos por separado el procedimiento de selección.
    from sklearn.model_selection import GridSearchCV
    import numpy as np
    pred_anidadas = np.empty(len(labels), dtype=object)
    for train, test in splits:
        # Dentro de cada entrenamiento, elegimos el clasificador sin mirar sus párrafos de prueba.
        cv_interna = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEED)
        busqueda = GridSearchCV(build_classifier(seed=SEED),
            {"classifier": [LogisticRegression(max_iter=1000, random_state=SEED), MultinomialNB()]},
            scoring="f1_macro", error_score="raise",
            cv=cv_interna)
        busqueda.fit(texts.iloc[train], labels.iloc[train], groups=groups.iloc[train])
        pred_anidadas[test] = busqueda.predict(texts.iloc[test])
    print(f"Validación anidada de la selección: accuracy = {accuracy_score(labels, pred_anidadas):.3f}; "
          f"F1 macro = {f1_score(labels, pred_anidadas, average='macro'):.3f}.")
    # Comparamos las dos representaciones sin cambiar las particiones.
    diferencia = 100 * (scores[1]["Accuracy"] - scores[0]["Accuracy"])
    print(f"Diferencia de accuracy con NLP: {diferencia:+.1f} puntos porcentuales.")
    display(pd.DataFrame(
        classification_report(labels, nlp_predictions, labels=LABELS, output_dict=True, zero_division=0)
    ).T.loc[LABELS].round(3))

    # Visualizamos los aciertos y las confusiones entre categorías.
    fig, ax = plt.subplots(figsize=(5.5, 4.4))
    ConfusionMatrixDisplay.from_predictions(
        labels, nlp_predictions, labels=LABELS, cmap="Blues", colorbar=False, ax=ax,
    )
    ax.set(xlabel="Predicción", ylabel="Etiqueta heredada", title="Matriz de confusión · modelo elegido")
    fig.tight_layout()
    plt.show()

    # Identificamos el error más frecuente entre clases distintas.
    errores = pd.crosstab(labels, pd.Series(nlp_predictions, index=labels.index))
    for label in LABELS:
        if label in errores.index and label in errores.columns:
            errores.loc[label, label] = 0
    if errores.to_numpy().max() > 0:
        real, predicha = errores.stack().idxmax()
        print(f"Confusión más frecuente: {real} → {predicha} ({errores.loc[real, predicha]} párrafos).")

    # Mostramos un ejemplo de error por clase para interpretar el resultado.
    ejemplos_error = data[["class", "article_title", "text"]].copy()
    ejemplos_error["Predicción"] = nlp_predictions
    ejemplos_error = ejemplos_error[ejemplos_error["class"] != ejemplos_error["Predicción"]]
    if not ejemplos_error.empty:
        ejemplos_error = ejemplos_error.groupby("class", group_keys=False).sample(n=1, random_state=SEED)
        with pd.option_context("display.max_colwidth", None):
            display(ejemplos_error.rename(columns={"class": "Etiqueta heredada", "article_title": "Artículo", "text": "Párrafo"}).reset_index(drop=True))

    print("Las discrepancias se miden frente a la etiqueta heredada. Pueden reflejar errores del modelo o ambigüedad temática.")

    # Entrenamos con todos los datos solo después de evaluar el modelo.
    model = build_classifier(use_nlp=True, seed=SEED, naive_bayes=configuraciones[mejor][2]).fit(texts, labels)
    matriz = model.named_steps["tfidf"].transform(texts)
    print(f"Matriz TF-IDF final: {matriz.shape[0]} párrafos × {matriz.shape[1]} características.")
    # Mostramos una pequeña parte de la matriz TF-IDF, con términos relevantes para tres párrafos.
    indices = data.groupby("class").head(1).index.to_numpy()
    columnas = []
    for indice in indices:
        fila = matriz[indice]
        # Seleccionamos dos términos de mayor peso por ejemplo.
        orden = fila.data.argsort()[::-1][:2]
        columnas.extend(fila.indices[orden])
    columnas = list(dict.fromkeys(columnas))
    nombres = model.named_steps["tfidf"].get_feature_names_out()
    if columnas:
        ejemplo_tfidf = pd.DataFrame(matriz[indices][:, columnas].toarray(),
                                     columns=nombres[columnas], index=labels.iloc[indices].to_list())
        display(ejemplo_tfidf.round(3))
    tiempo_modelo = time.perf_counter() - inicio
    print(f"Tiempo de evaluación y entrenamiento: {tiempo_modelo:.2f} s")

### **Elección del clasificador**

La regresión logística estima la probabilidad de cada categoría mediante los pesos de las características. Naive Bayes multinomial utiliza la frecuencia ponderada de las características y supone independencia condicional dada la categoría. Ambos son rápidos y admiten TF-IDF no negativo.

Usamos la regresión logística con y sin NLP para estudiar el efecto del preprocesamiento. Después elegimos entre los dos clasificadores con NLP según F1 macro. Como elegir a partir de las mismas cinco particiones puede dar una estimación optimista, mostramos también una validación anidada: dentro de cada entrenamiento comparamos los dos algoritmos y evaluamos el elegido en los artículos reservados. Esta cifra evalúa el procedimiento de selección, no un clasificador fijo.

La matriz de confusión y los ejemplos corresponden al modelo elegido en la comparación de cinco particiones. El vocabulario científico compartido explica parte de las discrepancias con las etiquetas heredadas.

### **Predicciones con probabilidades próximas**

El clasificador devuelve una probabilidad para cada categoría. Además de la predicción principal, señalamos los párrafos en los que las dos categorías más probables están a 5 puntos porcentuales o menos. Si las tres están a esa distancia, indicamos incertidumbre entre las tres.

Usamos las probabilidades obtenidas sobre el test de cada partición, no las del modelo entrenado con todo el corpus. Los 5 puntos son un criterio exploratorio, inspirado en la sugerencia del profesor, y no un umbral elegido para optimizar resultados. Este análisis no cambia ninguna predicción ni descarta párrafos: la accuracy, el F1 macro y la matriz de confusión anteriores se mantienen.

Las probabilidades de Naive Bayes no están necesariamente calibradas, por lo que las interpretamos como una señal relativa de duda. Que dos categorías tengan probabilidades próximas no demuestra que el párrafo sea realmente mixto: puede reflejar vocabulario compartido o falta de información en el texto.

In [ ]:
# Criterio exploratorio: distancia máxima entre las categorías más probables.
MARGEN_PROXIMO = 0.05
SIN_PROXIMIDAD = "Sin probabilidades próximas"


def describir_incertidumbre(probabilidades, margen=MARGEN_PROXIMO):
    """Indica si las dos o las tres categorías más probables están próximas."""
    orden = sorted(probabilidades, key=probabilidades.get, reverse=True)
    primera, segunda, tercera = (probabilidades[clase] for clase in orden)
    if primera - tercera <= margen:
        return "Incertidumbre entre las tres categorías"
    if primera - segunda <= margen:
        return f"Posible solapamiento: {orden[0]} y {orden[1]}"
    return SIN_PROXIMIDAD


if model is not None:
    # Analizamos las probabilidades de test del modelo elegido.
    proba = pd.DataFrame(nlp_probabilities, columns=LABELS, index=data.index)
    observacion = proba.apply(lambda fila: describir_incertidumbre(fila.to_dict()), axis=1)
    proximos = observacion != SIN_PROXIMIDAD
    acierto = pd.Series(nlp_predictions, index=data.index).eq(labels)

    resumen = pd.DataFrame({
        "Párrafos": [proximos.sum(), (~proximos).sum()],
        "% del corpus": [100 * proximos.mean(), 100 * (~proximos).mean()],
        "Errores": [(~acierto & proximos).sum(), (~acierto & ~proximos).sum()],
        "% de los errores": [100 * (~acierto & proximos).sum() / (~acierto).sum(),
                             100 * (~acierto & ~proximos).sum() / (~acierto).sum()],
        "Accuracy del grupo": [acierto[proximos].mean(), acierto[~proximos].mean()],
    }, index=["Probabilidades próximas", "Resto"])
    display(resumen.round(3))
    print(f"Accuracy global sin cambios: {acierto.mean():.1%}. "
          "La accuracy por grupo describe dónde se concentran los errores; no es una mejora del modelo.")

    # Vemos qué pares de categorías aparecen próximos y si la etiqueta heredada está entre ellos.
    dos_primeras = proba[proximos].apply(lambda fila: fila.nlargest(2).index.tolist(), axis=1)
    pares = dos_primeras.map(lambda par: " y ".join(sorted(par, key=LABELS.index)))
    display(pares.value_counts().rename_axis("Categorías próximas").to_frame("Párrafos"))
    incluida = sum(clase in par for clase, par in zip(labels[proximos], dos_primeras))
    print(f"La etiqueta heredada está entre las dos categorías más probables en {incluida} de {proximos.sum()} párrafos.")

    # Mostramos tres ejemplos reales, con aciertos y errores.
    ejemplos = data.loc[proximos, ["class", "article_title", "text"]].sample(n=min(3, proximos.sum()), random_state=SEED)
    ejemplos["Predicción"] = pd.Series(nlp_predictions, index=data.index)[ejemplos.index]
    ejemplos = ejemplos.join((proba.loc[ejemplos.index] * 100).round(1)).join(observacion.rename("Observación"))
    with pd.option_context("display.max_colwidth", None):
        display(ejemplos.rename(columns={"class": "Etiqueta heredada", "article_title": "Artículo", "text": "Párrafo"})
                .reset_index(drop=True))


### **Clasificación de páginas web**

Descargamos el HTML de tres artículos nuevos y extraemos sus párrafos con BeautifulSoup y aplicamos los mismos filtros de limpieza. Comprobamos que no estén en el conjunto de entrenamiento, aplicamos el mismo NLP y calculamos la media de las probabilidades de sus párrafos. Así obtenemos una predicción por página y, si las dos categorías más probables están a 5 puntos o menos, lo indicamos como observación. Si una descarga falla, continuamos con las restantes.

In [ ]:
"""Extracción de párrafos de páginas de Wikipedia y clasificación de su contenido."""

import re
from dataclasses import dataclass
from urllib.error import URLError
from urllib.request import Request, urlopen

from bs4 import BeautifulSoup


@dataclass(frozen=True)
class WebPage:
    page_id: int
    paragraphs: tuple[str, ...]


def parse_wikipedia_page(html, lang="eo"):
    """Recoge los párrafos del cuerpo del artículo y su identificador canónico."""
    # Analizamos el HTML y localizamos el cuerpo y el identificador del artículo.
    soup = BeautifulSoup(html, "html.parser")
    root = soup.html
    if root is None or root.get("lang") != lang:
        raise ValueError(f"El documento no declara el idioma {lang}")
    article = soup.select_one("#mw-content-text .mw-parser-output") or soup.select_one(".mw-parser-output")
    identifier = soup.find("meta", property="mw:pageId")
    if identifier is not None:
        page_id = int(identifier["content"])
    else:
        # Buscamos el ID en la configuración de MediaWiki si no aparece en los metadatos.
        config = re.search(r'"wgArticleId"\s*:\s*(\d+)', str(soup.head))
        page_id = int(config[1]) if config else 0
    if article is None or page_id <= 0:
        raise ValueError("No se encontró cuerpo de artículo con identificador de Wikipedia")

    # Retiramos elementos que no forman parte del texto principal.
    for element in article.select(
        "sup.reference, table, figure, .navbox, .infobox, .hatnote, .mw-editsection"
    ):
        element.decompose()

    lines = []
    for element in article.select("h2, h3, h4, h5, h6, p"):
        text = element.get_text(" ", strip=True)
        if element.name == "p":
            lines.append(text)
        else:
            marca = "=" * int(element.name[1])
            lines.append(f"{marca} {text} {marca}")
    paragraphs = tuple(dict.fromkeys(
        p for p in clean_paragraphs("\n".join(lines)) if not es_referencia_residual(p)
    ))
    if not paragraphs:
        raise ValueError("El artículo carece de párrafos utilizables")
    return WebPage(page_id, paragraphs)


def fetch_web_page(url, timeout=12):
    """Descarga HTML y devuelve el texto del artículo, o lanza un error legible."""
    request = Request(url, headers={"User-Agent": USER_AGENT, "Accept": "text/html"})
    try:
        with urlopen(request, timeout=timeout) as response:
            html = response.read()
    except (URLError, TimeoutError) as error:
        raise ValueError(f"Error al descargar la página: {error}") from error
    return parse_wikipedia_page(html)


def classify_web_page(model, page, training_article_ids, training_texts):
    """Promedia la probabilidad por clase sobre párrafos ajenos al corpus."""
    if page.page_id in training_article_ids:
        raise ValueError("El artículo forma parte del corpus de entrenamiento")
    paragraphs = [paragraph for paragraph in page.paragraphs if paragraph not in training_texts]
    if not paragraphs:
        raise ValueError("Todos los párrafos utilizables aparecen en el entrenamiento")
    # Promediamos las probabilidades de los párrafos nuevos.
    means = model.predict_proba(paragraphs).mean(axis=0)
    return len(paragraphs), dict(zip(model.classes_, means, strict=True))

In [ ]:
if model is not None:
    # Elegimos una página de prueba por categoría, ajena al entrenamiento.
    examples = PAGINAS_WEB
    training_ids = set(groups)
    training_paragraphs = set(texts)
    rows, failures = [], []

    # Descargamos y clasificamos cada página por separado.
    for title, expected in examples.items():
        url = "https://eo.wikipedia.org/wiki/" + quote(title.replace(" ", "_"))
        try:
            page = fetch_web_page(url)
            count, probabilities = classify_web_page(
                model, page, training_ids, training_paragraphs
            )
        except (ValueError, OSError, TypeError, KeyError, AttributeError) as error:
            failures.append({"Página": title, "Error": str(error)})
            continue

        rows.append({
            "Página": title,
            "Tema esperado": expected,
            "Predicción": max(probabilities, key=probabilities.get),
            "Párrafos": count,
            **probabilities,
            "Observación": describir_incertidumbre(probabilities),
        })

    if rows:
        resultados_web = pd.DataFrame(rows)
        display(resultados_web[["Página", "Tema esperado", "Predicción", "Párrafos"] + LABELS + ["Observación"]].round(3))
        # Comparamos visualmente las probabilidades medias de cada página.
        ax = (resultados_web.set_index("Página")[LABELS] * 100).plot.bar(
            figsize=(7, 3.5), rot=0, color=["#548c75", "#d1a24b", "#4878a8"])
        ax.set(title="Probabilidades medias por página", xlabel="", ylabel="Probabilidad (%)", ylim=(0, 100))
        ax.legend(title="Categoría")
        plt.tight_layout()
        plt.show()
        aciertos_web = resultados_web["Predicción"].eq(resultados_web["Tema esperado"]).sum()
        print(f"Coincidencias con el tema esperado: {aciertos_web}/{len(rows)} páginas descargadas.")
        menor = resultados_web.loc[resultados_web[LABELS].max(axis=1).idxmin()]
        print(f"La página con menor probabilidad máxima es {menor['Página']} "
              f"({menor[LABELS].max():.1%}, {menor['Párrafos']} párrafos). "
              "Estas probabilidades son medias del modelo, no garantías de acierto.")
    if failures:
        display(pd.DataFrame(failures))

## **4. Conclusiones**

Comparamos dos representaciones con regresión logística para estudiar la aportación de nuestro NLP, y después dos clasificadores con NLP. La validación anidada permite evaluar la selección sin utilizar las particiones externas para elegir el algoritmo. La matriz de confusión y los ejemplos de error permiten analizar las discrepancias con las etiquetas heredadas. Las probabilidades próximas entre categorías señalan un grupo reducido de párrafos que concentra una parte importante de los errores, sin cambiar las predicciones ni las métricas. Comparamos también el efecto de añadir artículos al entrenamiento, manteniendo fijos los artículos de prueba del corpus original; esta comparación se distingue de la validación del corpus completo.

La revisión del corpus muestra que la categoría del artículo no siempre representa el contenido de cada párrafo. Conservamos los textos científicos ambiguos y retiramos referencias bibliográficas, proverbios, apartados culturales y otros residuos. Algunos extractos de Wikipedia están cortados a mitad de frase; los conservamos si contienen información temática para no reconstruir textos ni alterar la muestra histórica. El indicador temático es orientativo: no mide por sí solo la tasa real de ruido. Además, algunas raíces son ambiguas: mas también coincide con maskloj (machos), kvant con kvanto (cantidad), fort con forta (fuerte) y baz con bazita (basado). Nuestro NLP también tiene limitaciones ante nombres propios, latinismos, fórmulas químicas, que tratamos como símbolos, y palabras cuya función depende del contexto. Reconocemos siglas en mayúsculas sin intentar lematizarlas como palabras flexionadas.

La clasificación de tres páginas nuevas es una demostración del uso del modelo, no una estimación de su rendimiento en todas las páginas web. Fijamos la semilla y usamos la versión de referencia de scikit-learn para favorecer la reproducibilidad. Los resultados pueden variar con otras versiones o al regenerar Wikipedia. El CSV de procedencia conserva los identificadores y revisiones de los artículos.

**Fuentes:** [Wikipedia en Esperanto](https://eo.wikipedia.org/) ([CC BY-SA](https://creativecommons.org/licenses/by-sa/4.0/)) y materiales de Minería de Textos: temas 0.4 (scikit-learn), 2.2 (NLP), 2.4 (clasificación de textos), 3.4 (web scraping) y 3.6 (API).


In [ ]:
if model is not None:
    # Resumimos los resultados que acabamos de obtener, sin escribir cifras a mano.
    resultado = scores[mejor]
    print(f"{configuraciones[mejor][0]}: accuracy = {resultado['Accuracy']:.1%}; F1 macro = {resultado['F1 macro']:.3f}.")
    print(f"Diferencia con NLP frente a TF-IDF directo (misma regresión logística): {diferencia:+.1f} puntos porcentuales de accuracy.")
    if diferencia > 0:
        print("En esta evaluación, el preprocesamiento con nuestro NLP mejora la clasificación.")
    else:
        print("En esta evaluación, el preprocesamiento con nuestro NLP no mejora la accuracy.")
    print(f"Evaluación y entrenamiento del clasificador: {tiempo_modelo:.2f} segundos.")
else:
    print("No podemos evaluar el modelo porque el corpus no está disponible.")